In [13]:
%%writefile PS4.cu
#include <stdio.h>
#include <cuda_runtime.h>

// CUDA kernel for 2D blocks and 2D threads
__global__ void helloFromThreads2D() {
    // 2D thread index within its block
    int threadX = threadIdx.x;
    int threadY = threadIdx.y;

    // 2D block index within the grid
    int blockX = blockIdx.x;
    int blockY = blockIdx.y;

    // Calculate a unique global thread index in the grid
    int globalX = blockX * blockDim.x + threadX;
    int globalY = blockY * blockDim.y + threadY;

    printf("Hello World from block(%d,%d), thread(%d,%d) → global(%d,%d)\n",
           blockX, blockY, threadX, threadY, globalX, globalY);
}

int main() {
    dim3 threadsPerBlock(4, 2); // 4 threads in x, 2 in y per block
    dim3 numBlocks(2, 2);       // 2 blocks in x, 2 in y grid

    helloFromThreads2D<<<numBlocks, threadsPerBlock>>>();

    cudaDeviceSynchronize();

    return 0;
}

Overwriting PS4.cu


In [15]:
!nvcc -arch=sm_75 PS4.cu -o PS4

In [16]:
!./PS4

Hello World from block(1,1), thread(0,0) → global(4,2)
Hello World from block(1,1), thread(1,0) → global(5,2)
Hello World from block(1,1), thread(2,0) → global(6,2)
Hello World from block(1,1), thread(3,0) → global(7,2)
Hello World from block(1,1), thread(0,1) → global(4,3)
Hello World from block(1,1), thread(1,1) → global(5,3)
Hello World from block(1,1), thread(2,1) → global(6,3)
Hello World from block(1,1), thread(3,1) → global(7,3)
Hello World from block(0,0), thread(0,0) → global(0,0)
Hello World from block(0,0), thread(1,0) → global(1,0)
Hello World from block(0,0), thread(2,0) → global(2,0)
Hello World from block(0,0), thread(3,0) → global(3,0)
Hello World from block(0,0), thread(0,1) → global(0,1)
Hello World from block(0,0), thread(1,1) → global(1,1)
Hello World from block(0,0), thread(2,1) → global(2,1)
Hello World from block(0,0), thread(3,1) → global(3,1)
Hello World from block(0,1), thread(0,0) → global(0,2)
Hello World from block(0,1), thread(1,0) → global(1,2)
Hello Worl

In [1]:
%%writefile PS5.cu
#include <stdio.h>
#include <stdlib.h>
#include <cuda.h>
#include <time.h>

// CUDA kernel for vector addition
__global__ void vectorAddKernel(const float *A, const float *B, float *C, int N) {
    int globalIndex = blockIdx.x * blockDim.x + threadIdx.x; // global thread ID
    if (globalIndex < N)
        C[globalIndex] = A[globalIndex] + B[globalIndex];
}

// Initialize host vectors with random values
void initializeVectors(float *A, float *B, int N) {
    for (int i = 0; i < N; i++) {
        A[i] = (float)(rand() % 100) / 10.0f;
        B[i] = (float)(rand() % 100) / 10.0f;
    }
}

// CPU implementation of vector addition
void vectorAddCPU(const float *A, const float *B, float *C, int N) {
    for (int i = 0; i < N; i++)
        C[i] = A[i] + B[i];
}

int main() {
    int N = 1e6;  // 1 million elements
    size_t bytes = N * sizeof(float);

    // Host (CPU) memory allocation
    float *h_A = (float *)malloc(bytes);
    float *h_B = (float *)malloc(bytes);
    float *h_C_CPU = (float *)malloc(bytes);
    float *h_C_GPU = (float *)malloc(bytes);

    srand(time(NULL));
    initializeVectors(h_A, h_B, N);

    // --- CPU computation ---
    clock_t startCPU = clock();
    vectorAddCPU(h_A, h_B, h_C_CPU, N);
    clock_t endCPU = clock();
    double cpuTime = (double)(endCPU - startCPU) / CLOCKS_PER_SEC;

    // --- GPU computation setup ---
    float *d_A, *d_B, *d_C;
    cudaMalloc((void **)&d_A, bytes);
    cudaMalloc((void **)&d_B, bytes);
    cudaMalloc((void **)&d_C, bytes);

    cudaMemcpy(d_A, h_A, bytes, cudaMemcpyHostToDevice);
    cudaMemcpy(d_B, h_B, bytes, cudaMemcpyHostToDevice);

    int threadsPerBlock = 256;
    int numBlocks = (N + threadsPerBlock - 1) / threadsPerBlock;

    cudaEvent_t startGPU, stopGPU;
    cudaEventCreate(&startGPU);
    cudaEventCreate(&stopGPU);
    cudaEventRecord(startGPU);

    // --- Launch kernel ---
    vectorAddKernel<<<numBlocks, threadsPerBlock>>>(d_A, d_B, d_C, N);

    cudaEventRecord(stopGPU);
    cudaEventSynchronize(stopGPU);

    cudaMemcpy(h_C_GPU, d_C, bytes, cudaMemcpyDeviceToHost);

    float gpuTime = 0.0f;
    cudaEventElapsedTime(&gpuTime, startGPU, stopGPU);
    gpuTime /= 1000.0f;  // convert ms to seconds

    // --- Results ---
    printf("\n=== Vector Addition using CUDA ===\n");
    printf("Number of Elements: %d\n", N);
    printf("CPU Execution Time: %.6f s\n", cpuTime);
    printf("GPU Execution Time: %.6f s\n", gpuTime);
    printf("Speedup (CPU/GPU): %.2fx\n", cpuTime / gpuTime);

    // --- Cleanup ---
    free(h_A);
    free(h_B);
    free(h_C_CPU);
    free(h_C_GPU);
    cudaFree(d_A);
    cudaFree(d_B);
    cudaFree(d_C);

    return 0;
}

Writing PS5.cu


In [2]:
!nvcc -arch=sm_75 PS5.cu -o PS5

In [3]:
!./PS5


=== Vector Addition using CUDA ===
Number of Elements: 1000000
CPU Execution Time: 0.004470 s
GPU Execution Time: 0.000153 s
Speedup (CPU/GPU): 29.15x


In [4]:
%%writefile PS6.cu
#include <stdio.h>
#include <stdlib.h>
#include <cuda.h>
#include <time.h>

__global__ void matrixAdd(float *A, float *B, float *C, int M, int N) {
    int row = blockIdx.y * blockDim.y + threadIdx.y;
    int col = blockIdx.x * blockDim.x + threadIdx.x;
    int idx = row * N + col;
    if (row < M && col < N)
        C[idx] = A[idx] + B[idx];
}

void initializeMatrix(float *A, float *B, int M, int N) {
    for (int i = 0; i < M * N; i++) {
        A[i] = (float)(rand() % 100) / 10.0;
        B[i] = (float)(rand() % 100) / 10.0;
    }
}

void matrixAddCPU(float *A, float *B, float *C, int M, int N) {
    for (int i = 0; i < M * N; i++)
        C[i] = A[i] + B[i];
}

int main() {
    int M = 1000, N = 1000;
    size_t size = M * N * sizeof(float);

    float *h_A = (float *)malloc(size);
    float *h_B = (float *)malloc(size);
    float *h_C_cpu = (float *)malloc(size);
    float *h_C_gpu = (float *)malloc(size);

    srand(time(NULL));
    initializeMatrix(h_A, h_B, M, N);

    // --- CPU computation ---
    clock_t start_cpu = clock();
    matrixAddCPU(h_A, h_B, h_C_cpu, M, N);
    clock_t end_cpu = clock();
    double cpu_time = ((double)(end_cpu - start_cpu)) / CLOCKS_PER_SEC;

    // --- GPU computation setup ---
    float *d_A, *d_B, *d_C;
    cudaMalloc((void **)&d_A, size);
    cudaMalloc((void **)&d_B, size);
    cudaMalloc((void **)&d_C, size);

    cudaMemcpy(d_A, h_A, size, cudaMemcpyHostToDevice);
    cudaMemcpy(d_B, h_B, size, cudaMemcpyHostToDevice);

    dim3 blockSize(16, 16);
    dim3 numBlocks((N + blockSize.x - 1) / blockSize.x,
                   (M + blockSize.y - 1) / blockSize.y);

    cudaEvent_t start_gpu, stop_gpu;
    cudaEventCreate(&start_gpu);
    cudaEventCreate(&stop_gpu);
    cudaEventRecord(start_gpu);

    // --- Kernel launch ---
    matrixAdd<<<numBlocks, blockSize>>>(d_A, d_B, d_C, M, N);

    cudaEventRecord(stop_gpu);
    cudaEventSynchronize(stop_gpu);

    cudaMemcpy(h_C_gpu, d_C, size, cudaMemcpyDeviceToHost);

    float gpu_time = 0;
    cudaEventElapsedTime(&gpu_time, start_gpu, stop_gpu);
    gpu_time /= 1000.0;

    // --- Print performance results ---
    printf("\n=== Matrix Addition (GPU vs CPU) ===\n");
    printf("Matrix Size: %dx%d\n", M, N);
    printf("CPU Time: %.6f s\n", cpu_time);
    printf("GPU Time: %.6f s\n", gpu_time);
    printf("Speedup (CPU/GPU): %.2fx\n", cpu_time / gpu_time);

    // --- Cleanup ---
    free(h_A);
    free(h_B);
    free(h_C_cpu);
    free(h_C_gpu);
    cudaFree(d_A);
    cudaFree(d_B);
    cudaFree(d_C);

    return 0;
}

Writing PS6.cu


In [5]:
!nvcc -arch=sm_75 PS6.cu -o PS6

In [6]:
!./PS6


=== Matrix Addition (GPU vs CPU) ===
Matrix Size: 1000x1000
CPU Time: 0.004315 s
GPU Time: 0.000094 s
Speedup (CPU/GPU): 45.80x


In [7]:
%%writefile PS7.cu
#include <stdio.h>
#include <stdlib.h>
#include <cuda.h>
#include <time.h>

// CUDA kernel to compute partial dot products
__global__ void dotProductKernel(float *A, float *B, float *result, int N) {
    __shared__ float cache[256];  // shared memory for reduction
    int tid = blockIdx.x * blockDim.x + threadIdx.x;
    int cacheIndex = threadIdx.x;

    float temp = 0.0f;

    // Each thread computes part of the dot product
    while (tid < N) {
        temp += A[tid] * B[tid];
        tid += blockDim.x * gridDim.x; // move to next element handled by same thread
    }

    cache[cacheIndex] = temp;  // store in shared memory
    __syncthreads();

    // Reduction within the block
    int i = blockDim.x / 2;
    while (i != 0) {
        if (cacheIndex < i)
            cache[cacheIndex] += cache[cacheIndex + i];
        __syncthreads();
        i /= 2;
    }

    // Add the block's result to the global result
    if (cacheIndex == 0)
        atomicAdd(result, cache[0]);
}

// Initialize vectors with random float values
void initializeVectors(float *A, float *B, int N) {
    for (int i = 0; i < N; i++) {
        A[i] = (float)(rand() % 100) / 10.0f;
        B[i] = (float)(rand() % 100) / 10.0f;
    }
}

// CPU reference computation
float dotProductCPU(float *A, float *B, int N) {
    float sum = 0.0f;
    for (int i = 0; i < N; i++)
        sum += A[i] * B[i];
    return sum;
}

int main() {
    int N = 100000;  // vector size
    size_t size = N * sizeof(float);

    float *h_A = (float *)malloc(size);
    float *h_B = (float *)malloc(size);
    float h_result_cpu = 0.0f, h_result_gpu = 0.0f;

    srand(time(NULL));
    initializeVectors(h_A, h_B, N);

    // --- CPU computation ---
    clock_t start_cpu = clock();
    h_result_cpu = dotProductCPU(h_A, h_B, N);
    clock_t end_cpu = clock();
    double cpu_time = ((double)(end_cpu - start_cpu)) / CLOCKS_PER_SEC;

    // --- GPU computation ---
    float *d_A, *d_B, *d_result;
    cudaMalloc((void **)&d_A, size);
    cudaMalloc((void **)&d_B, size);
    cudaMalloc((void **)&d_result, sizeof(float));

    cudaMemcpy(d_A, h_A, size, cudaMemcpyHostToDevice);
    cudaMemcpy(d_B, h_B, size, cudaMemcpyHostToDevice);
    cudaMemcpy(d_result, &h_result_gpu, sizeof(float), cudaMemcpyHostToDevice);

    int blockSize = 256;
    int numBlocks = 256;

    cudaEvent_t start_gpu, stop_gpu;
    cudaEventCreate(&start_gpu);
    cudaEventCreate(&stop_gpu);
    cudaEventRecord(start_gpu);

    dotProductKernel<<<numBlocks, blockSize>>>(d_A, d_B, d_result, N);
    cudaDeviceSynchronize();

    cudaEventRecord(stop_gpu);
    cudaEventSynchronize(stop_gpu);

    cudaMemcpy(&h_result_gpu, d_result, sizeof(float), cudaMemcpyDeviceToHost);

    float gpu_time = 0.0f;
    cudaEventElapsedTime(&gpu_time, start_gpu, stop_gpu);
    gpu_time /= 1000.0f; // convert ms to seconds

    // --- Results ---
    printf("\n=== Dot Product Computation ===\n");
    printf("Vector Size: %d\n", N);
    printf("CPU Result: %.4f\n", h_result_cpu);
    printf("GPU Result: %.4f\n", h_result_gpu);
    printf("CPU Time: %.6f s\n", cpu_time);
    printf("GPU Time: %.6f s\n", gpu_time);
    printf("Speedup (CPU/GPU): %.2fx\n", cpu_time / gpu_time);

    // --- Cleanup ---
    free(h_A);
    free(h_B);
    cudaFree(d_A);
    cudaFree(d_B);
    cudaFree(d_result);

    return 0;
}

Writing PS7.cu


In [8]:
!nvcc -arch=sm_75 PS7.cu -o PS7

In [9]:
!./PS7


=== Dot Product Computation ===
Vector Size: 100000
CPU Result: 2456819.2500
GPU Result: 2456815.0000
CPU Time: 0.000291 s
GPU Time: 0.000120 s
Speedup (CPU/GPU): 2.43x


In [10]:
%%writefile PS8.cu
#include <stdio.h>
#include <stdlib.h>
#include <cuda.h>
#include <time.h>

// GPU Kernel: Matrix Multiplication
__global__ void matrixMultiply(float *A, float *B, float *C, int M, int N, int P) {
    int row = blockIdx.y * blockDim.y + threadIdx.y;  // Row index of C
    int col = blockIdx.x * blockDim.x + threadIdx.x;  // Column index of C

    if (row < M && col < P) {
        float sum = 0.0f;
        for (int k = 0; k < N; ++k)
            sum += A[row * N + k] * B[k * P + col];
        C[row * P + col] = sum;
    }
}

// Initialize a matrix with random float values
void initializeMatrix(float *A, int rows, int cols) {
    for (int i = 0; i < rows * cols; ++i)
        A[i] = (float)(rand() % 100) / 10.0f;
}

// CPU function for matrix multiplication
void matrixMultiplyCPU(float *A, float *B, float *C, int M, int N, int P) {
    for (int i = 0; i < M; ++i) {
        for (int j = 0; j < P; ++j) {
            float sum = 0.0f;
            for (int k = 0; k < N; ++k)
                sum += A[i * N + k] * B[k * P + j];
            C[i * P + j] = sum;
        }
    }
}

int main() {
    // Define matrix dimensions: A(MxN), B(NxP), C(MxP)
    int M = 500, N = 500, P = 500;
    size_t size_A = M * N * sizeof(float);
    size_t size_B = N * P * sizeof(float);
    size_t size_C = M * P * sizeof(float);

    // Allocate host memory
    float *h_A = (float *)malloc(size_A);
    float *h_B = (float *)malloc(size_B);
    float *h_C_cpu = (float *)malloc(size_C);
    float *h_C_gpu = (float *)malloc(size_C);

    // Initialize input matrices
    srand(time(NULL));
    initializeMatrix(h_A, M, N);
    initializeMatrix(h_B, N, P);

    // --- CPU Computation ---
    clock_t start_cpu = clock();
    matrixMultiplyCPU(h_A, h_B, h_C_cpu, M, N, P);
    clock_t end_cpu = clock();
    double cpu_time = ((double)(end_cpu - start_cpu)) / CLOCKS_PER_SEC;

    // --- GPU Computation ---
    float *d_A, *d_B, *d_C;
    cudaMalloc((void **)&d_A, size_A);
    cudaMalloc((void **)&d_B, size_B);
    cudaMalloc((void **)&d_C, size_C);

    cudaMemcpy(d_A, h_A, size_A, cudaMemcpyHostToDevice);
    cudaMemcpy(d_B, h_B, size_B, cudaMemcpyHostToDevice);

    // Configure grid and block dimensions
    dim3 blockSize(16, 16);
    dim3 numBlocks((P + blockSize.x - 1) / blockSize.x,
                   (M + blockSize.y - 1) / blockSize.y);

    cudaEvent_t start_gpu, stop_gpu;
    cudaEventCreate(&start_gpu);
    cudaEventCreate(&stop_gpu);
    cudaEventRecord(start_gpu);

    // Launch kernel
    matrixMultiply<<<numBlocks, blockSize>>>(d_A, d_B, d_C, M, N, P);
    cudaDeviceSynchronize();

    cudaEventRecord(stop_gpu);
    cudaEventSynchronize(stop_gpu);

    // Copy result back to host
    cudaMemcpy(h_C_gpu, d_C, size_C, cudaMemcpyDeviceToHost);

    // Measure GPU time
    float gpu_time = 0.0f;
    cudaEventElapsedTime(&gpu_time, start_gpu, stop_gpu);
    gpu_time /= 1000.0f; // Convert ms to seconds

    // --- Results ---
    printf("\n=== Matrix Multiplication (CPU vs GPU) ===\n");
    printf("Matrix A: %dx%d\n", M, N);
    printf("Matrix B: %dx%d\n", N, P);
    printf("Matrix C: %dx%d\n", M, P);
    printf("-----------------------------------------\n");
    printf("CPU Time: %.6f s\n", cpu_time);
    printf("GPU Time: %.6f s\n", gpu_time);
    printf("Speedup (CPU/GPU): %.2fx\n", cpu_time / gpu_time);

    // --- Cleanup ---
    free(h_A);
    free(h_B);
    free(h_C_cpu);
    free(h_C_gpu);
    cudaFree(d_A);
    cudaFree(d_B);
    cudaFree(d_C);

    return 0;
}

Writing PS8.cu


In [11]:
!nvcc -arch=sm_75 PS8.cu -o PS8

In [12]:
!./PS8


=== Matrix Multiplication (CPU vs GPU) ===
Matrix A: 500x500
Matrix B: 500x500
Matrix C: 500x500
-----------------------------------------
CPU Time: 0.472350 s
GPU Time: 0.001091 s
Speedup (CPU/GPU): 433.11x
